# Dynamic mechanical fragmentation — theory and numerical scheme

This notebook documents the physical model, its variational structure, the
discretisation and the algorithm implemented in `problems/dynamic.py`.

**Contents**

1. Problem setup (geometry, loading, boundary conditions)
2. Energy functional (elastic, foundation, fracture, kinetic)
3. AT1 vs AT2 dissipation densities
4. Quasi-static formulation: alternate minimisation
5. Dynamic formulation: Newmark-$\beta$ + alternate minimisation
6. Spatial discretisation (FEM)
7. Bound-constrained damage solver (PETSc VINEWTONRSLS)
8. Algorithm pseudo-code
9. Quantities of interest

## 1.  Problem setup

We consider a homogeneous body $\Omega$ in dimension $d \in\{1,2\}$ resting on a linear elastic foundation of stiffness $\Lambda^{2}$.  The body is loaded *kinematically* at its right end by a prescribed displacement that ramps up smoothly in time.

- **1D**:  $\Omega = (0, L_x)$.
- **2D**:  $\Omega = (0, L_x)\times(0, L_y)$, meshed with triangles.

Boundary conditions (in dimension $d$):

$$
\boldsymbol{u}(0,\cdot,t) = \boldsymbol{0},
\qquad
u_x(L_x,\cdot,t) = \hat U(t),
\qquad
\alpha(0,\cdot,t)=\alpha(L_x,\cdot,t)=0.
$$

The smooth loading ramp

$$
\hat U(t) \;=\; \dot U_p\,\bigl(\sqrt{T_0^{\,2} + t^{2}} - T_0\bigr),
\qquad
\dot U_p := \dfrac{\hat U_{\max}}{\sqrt{T_0^{\,2}+1}-T_0},
$$

is $C^{\infty}$ on $[0,1]$, starts with zero velocity and reaches $\hat U_{\max}$ at $t=1$.  $T_0$ controls how fast the velocity ramps up: $T_0 \to 0$ recovers a Heaviside-like jump, $T_0 \to\infty$ recovers a linear ramp.

## 2.  Energy functional

We work with a single damage variable $\alpha\in[0,1]$ and a displacement field $\boldsymbol{u}$.  The total potential energy is

$$
\mathcal{E}(\boldsymbol{u},\alpha)
\;=\;
\underbrace{\int_\Omega \tfrac{1}{2}\,g(\alpha)\,\boldsymbol{\varepsilon}(\boldsymbol{u}):\mathbb{C}:\boldsymbol{\varepsilon}(\boldsymbol{u})\,d\Omega}_{\mathcal{P}_{\!el}}
\;+\;
\underbrace{\int_\Omega \tfrac{1}{2}\,\Lambda^{2}\,|\boldsymbol{u}|^{2}\,d\Omega}_{\mathcal{P}_{\!f}}
\;+\;
\underbrace{\int_\Omega \psi_d(\alpha,\nabla\alpha)\,d\Omega}_{\mathcal{S}}.
$$

With kinetic energy

$$
\mathcal{K}(\dot{\boldsymbol{u}}) \;=\; \int_\Omega \tfrac{1}{2}\,\eta^{2}\,|\dot{\boldsymbol{u}}|^{2}\,d\Omega,
$$

the Lagrangian is $\mathcal{L} = \mathcal{K}-\mathcal{E}$ and the action $\int_0^T \mathcal{L}\,dt$ is stationary along the trajectory of the system, supplemented by the dissipation inequality $\dot\alpha\ge 0$ for the damage.

**Notation.**
- $g(\alpha)=(1-\alpha)^{2}$ is the elastic degradation function.
- In 1D the strain is the scalar $\varepsilon = \partial u/\partial x$ and $\mathbb{C}=1$.
- In 2D the strain is the symmetric tensor $\varepsilon = \tfrac12(\nabla\boldsymbol{u}+\nabla\boldsymbol{u}^\top)$; in this repository we use the unit shear modulus $\mu=1$, $\lambda_{\text{Lame}}=0$ so that $\mathbb{C}:\varepsilon = 2\varepsilon$.
- $\eta$ is a non-dimensional inverse wave speed.  $\eta\to 0$ recovers the quasi-static limit; $\eta\sim 1$ is the inertia-dominated regime.

## 3.  Fracture-energy density: AT1 vs AT2

Following Bourdin, Marigo and Maurini, we parametrise the dissipation by

$$
\psi_d(\alpha,\nabla\alpha) \;=\; \frac{G_c}{c_w}\,\Bigl(\,\frac{w(\alpha)}{\ell}\;+\;\ell\,|\nabla\alpha|^{2}\,\Bigr).
$$

After non-dimensionalisation ($G_c/c_w = 1$, $\ell$ replaced by $\hat\ell$, lengths by $L_x$) the discrete form coded in `tools/solvers.py` is

$$
\hat\psi_d \;=\; \frac{1}{c_w}\bigl(w(\alpha) + \hat\ell^{\,2}|\nabla\alpha|^{2}\bigr).
$$

Two variants are shipped:

| Variant | $w(\alpha)$ | $c_w$ | Sub-critical phase? |
|---|---|---|---|
| **AT1** | $\alpha$  | $8/3$ | yes — damage stays zero until a critical strain is reached |
| **AT2** | $\alpha^2$| $2$   | no — damage grows continuously from $t = 0^+$ |

Switching between AT1 and AT2 is the only change needed to add a new model: simply add an entry to the `MODELS` dictionary in `tools/solvers.py`.

## 4.  Quasi-static formulation

In the quasi-static limit ($\eta\to 0$) the inertia term is dropped and we look for $(\boldsymbol{u}^n,\alpha^n)$ at time $t^n\in(0,1]$ as a local minimiser of $\mathcal{E}$ subject to:

- the kinematic constraint $u_x(L_x) = \hat U(t^n)$,
- the irreversibility constraint $\alpha^n \ge \alpha^{n-1}$ (cracks cannot heal),
- the bounds $0\le\alpha^n\le 1$.

The non-convexity of $\mathcal{E}$ is handled by **alternate minimisation** (AltMin): keep one field fixed, minimise over the other, swap, repeat.

**Inner loop at fixed $t^n$.**
$$
\begin{aligned}
\boldsymbol{u}^{n,k+1} &= \arg\min_{\boldsymbol{u}\,\text{adm.}}
  \mathcal{E}\bigl(\boldsymbol{u},\alpha^{n,k}\bigr)
  &&\text{(linear elliptic problem)}\\
\alpha^{n,k+1} &= \arg\min_{\alpha^{n-1}\le\alpha\le 1}
  \mathcal{E}\bigl(\boldsymbol{u}^{n,k+1},\alpha\bigr)
  &&\text{(bound-constrained QP)}\\
&\text{stop when } \|\alpha^{n,k+1}-\alpha^{n,k}\|_{L^{2}}\le \mathrm{tol}.
\end{aligned}
$$

Once converged we set $\alpha^{n}=\alpha^{n,k+1}$ and update the irreversibility lower bound $\underline{\alpha}\leftarrow\alpha^{n}$.

## 5.  Dynamic formulation — Newmark-$\beta$

The Euler-Lagrange equations of $\mathcal{K}-\mathcal{E}$ read

$$
\eta^{2}\,\ddot{\boldsymbol{u}}\;-\;\nabla\!\cdot\!\bigl(g(\alpha)\,\mathbb{C}:\boldsymbol{\varepsilon}\bigr)\;+\;\Lambda^{2}\,\boldsymbol{u} \;=\; \boldsymbol{0}
$$

together with the damage variational inequality

$$
\dot\alpha \ge 0,\qquad
\partial_\alpha\mathcal{E}\ge 0,\qquad
\dot\alpha\,\partial_\alpha\mathcal{E} = 0.
$$

Time integration uses the implicit **Newmark-$\beta$** scheme:

$$
\boldsymbol{u}^{n+1} = \boldsymbol{u}^{n} + \Delta t\,\boldsymbol{v}^{n} + \tfrac12\Delta t^{2}\bigl[(1-2\beta)\boldsymbol{a}^{n} + 2\beta\,\boldsymbol{a}^{n+1}\bigr],
$$
$$
\boldsymbol{v}^{n+1} = \boldsymbol{v}^{n} + \Delta t\,\bigl[(1-\gamma)\boldsymbol{a}^{n} + \gamma\,\boldsymbol{a}^{n+1}\bigr].
$$

We use the *average acceleration* values $\beta=1/4$, $\gamma=1/2$, which is **unconditionally stable** for linear problems.

Within one Newmark step we still alternate-minimise on $(\boldsymbol{a}^{n+1},\alpha^{n+1})$: substituting the Newmark update of $\boldsymbol{u}^{n+1}$ into $\mathcal{E}$ gives a linear equation for $\boldsymbol{a}^{n+1}$ at fixed $\alpha$, then a bound-constrained QP for $\alpha^{n+1}$ at fixed $\boldsymbol{a}^{n+1}$.  The velocity and displacement are reconstructed at the end of the step.

## 6.  Spatial discretisation

All fields are discretised with continuous **piece-wise linear** Lagrange elements (P1):

- $V_u$ — scalar P1 in 1D, vector P1 in 2D.
- $V_\alpha$ — scalar P1.

The 2D mesh is generated with `dolfinx.mesh.create_rectangle(..., cell_type=CellType.triangle, diagonal=DiagonalType.crossed)`, which splits every rectangular cell into four triangles.  The resulting mesh is symmetric in $x$ and $y$, removing the spurious anisotropy of the standard `right`/`left` diagonals — important for studying *fragmentation patterns*.

A safe rule of thumb is to keep the mesh resolved with respect to the internal length:

$$
h \;\lesssim\; \hat\ell / 2
$$

so that each diffuse crack band is resolved by at least four elements across its width.

## 7.  Bound-constrained damage solver

The damage minimisation step is a **variational inequality**:

$$
\underline{\alpha} \;\le\; \alpha \;\le\; 1,
\qquad
\partial_\alpha\mathcal{E}\cdot(\beta-\alpha)\ge 0\quad\forall\,\beta\in[\underline{\alpha},1].
$$

PETSc's [`SNESVINEWTONRSLS`](https://petsc.org/release/manualpages/SNES/SNESVINEWTONRSLS/) solves it as a *reduced-space active-set Newton* method:

1. Predict the active set $\{\alpha = \underline{\alpha}\}\cup\{\alpha = 1\}$.
2. Solve a Newton step in the *reduced* (inactive) sub-space.
3. Project back to $[\underline{\alpha},1]$ and update the active set.

Bounds are passed via `setVariableBounds(alpha_lb, alpha_ub)`.  After every converged step we update $\underline{\alpha}\leftarrow\alpha$ so that the *next* step inherits the irreversibility constraint.

## 8.  Algorithm pseudo-code

```
build mesh, P1 spaces, boundary conditions      # tools.meshing
assemble energy densities + AT1/AT2 dissipation # tools.solvers

# QUASI-STATIC LOOP
for ti in (t1, ..., tN_qs):
    u_right = U(ti)
    repeat
        solve elastic equilibrium  (SNES ksponly+LU)
        solve damage VI            (SNES vinewtonrsls+LU)
    until || alpha - alpha_prev ||_L2 < tol
    alpha_lb := alpha
    record reaction, energies

# DYNAMIC LOOP
u, v, a = 0
for step in 1..N_dyn:
    t += dt
    u_right, v_right, a_right = U(t), V(t), A(t)
    repeat
        solve linear problem for a_new     (Newmark + SNES ksponly)
        solve damage VI                    (SNES vinewtonrsls)
    until convergence
    u_new = u + dt v + 0.5 dt^2 ((1-2 beta) a + 2 beta a_new)
    v_new = v + dt ((1-gamma) a + gamma a_new)
    advance u, v, a; update alpha_lb
    record reaction + energies
    if step in snapshots: append (alpha, u) to Paraview series
```

The inner *repeat ... until convergence* block is `tools.helpers.alt_min_loop`.

## 9.  Quantities of interest

For each run the code records, at every time step,

- $\hat U(t)$ and the reaction force $\hat F(t) = \int_{x=L_x} g(\alpha)\,\sigma_{xx}\,dS$;
- the **elastic** energy $\mathcal{P}_{\!el}$, **foundation** energy $\mathcal{P}_{\!f}$ and **fracture** energy $\mathcal{S}$;
- the kinetic energy $\mathcal{K}$ (dynamic run only);
- the total $\mathcal{E}+\mathcal{K}$.

Post-processing draws three sub-plots (`tools/plotting.plot_mechanical_run`):

1. force–displacement curve (QS vs dyn);
2. final damage profile (1D) or `tripcolor` of $\alpha$ (2D);
3. all energy histories overlaid, plus the inf-norm gap between QS and dyn — a useful indicator of inertial overshoot.

The Paraview series (XDMF) contains $\alpha$ and $\boldsymbol u$ at user-chosen snapshots; it is what we use to look at crack-pattern movies in 2D.

## How to run

```bash
# Single run, 1D, AT2:
python problems/dynamic.py

# 2D variant: open problems/dynamic.py and set
#   cfg['mesh_parameters']['physics']  = '2D'
#   cfg['solver_parameters']['model']  = 'AT1'
# then re-run.

# Parameter sweep across (l_hat, Lambda, eta, model, physics):
python sweep.py dynamic
```